In [3]:
# STRUCTURE:
# actual results, and initial stop (if never moved)
# no stop
# wide stop / max tolerable loss (20-40% range)
# 8, 10, 12.5, 15% stops
# 2, 2.5, 3 ATR stops
# maybe daily higher low and weekly higher low pivot stops, if I can figure that out, in a more advanced version

# ASSUMPTIONS:
# Entry is taken around the close of the day, so the first day is skipped when assessing R-multiples. Hence df.iloc[1:] syntax.
# On days where price gaps down below my stop, the stop is triggered at the open. Not always going to be the case
# No commissions or slippage on exit, so a round -1R loss if stop is hit intraday. Would like to fix in a future version.
# Intraday action is currently ignored. This may be an issue on days where my stop hits before a new high for the move. Will probably address with intraday price data and resampling. 

# EDGE CASES:
# Think about intraday entry and exit timing. E.g. what if my stop is below the low of the day, but I enter after the low of the day is set? Does it matter if entry is at the close?
# what if there's no data from df.iloc[1:]? 
# what should realised_r return if not stopped out? na value?
# what happens on a day if my stop hits before the high of the day? 
# Using business days is better than calendar days for the pandas offset, but market holidays would be an issue. Will accept limitation for now. 

# FUTURE CONSIDERATIONS:
# More generally, I could do with error-checking to make the program more robust. This should be built alongside the functions, rather than being an afterthought. 
# Would I go about this process a different way with a larger dataset? Does pandas have a built in function, so I don't have to use the slower python loops?
# Instead of looping yfinance for every trade, should I store price data in a csv? Or would that be unnecessary? Depends on speed, data accuracy, etc. 
# Should I add some kind of drawdown calculation in a future version, so I can see what type of pullbacks I might expect and how viable wide stops actually are.
# While this isn't a consideration for v1, for future versions, I'd like to consider intraday stops, slippage, commissions,etc. I want to make this professional-grade.
# At some point I plan to test trailing stops on profitable trades to see what the most effective method there would be. E.g. pivot lows, moving average, atr, percentage trailing, etc.
# Worth adding MAE and MFE at some stage, to see how much a position goes for and against me before resolution. 
# I think I should replace exit data and exit price with last_date and last_price (or final) and then add a boolean column for stopped_out (True/False), would help with available_r. 
# no_stop_max_r can be derived from this using the trade method's 1r value, if desired...
# Add realised_r (or available_r for open trades) and max_r back for each trade, and think about no_stop_max_r comparisons.
# I think max drawdown is going to be important when comparing stop loss types. E.g. A wider stop might look better on paper, but is the drop tolerable psychologically?

# TASKS:
# Create atr_stop functions, and repeat pandas analysis. 
# Try figuring out the daily and weekly pivot stops. If it's not getting anywhere, save for future version and move on to Pardo.
# Perhaps worth considering EV or risk-adjusted returns using r-multiples calculations. 
# Could replace the iterrow loops with some vectorised form, such as np.where or pandas cummax. 

# ISSUES:
# Minor inconsistency in functions and order of ticker/date return. Not a meaningful concern.
# Probably better to use start date + 1 business day across the board rather than iloc[1:], but not a major issue for now. 

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [5]:
# pandas loop helper function

def trade_loop(data, entry_price, stop_price):

    max_price = entry_price
    exit_date = np.nan
    exit_price = np.nan

    for row_name, row in data.iloc[1:].iterrows():

        if row['High'] > max_price:
            max_price = row['High']

        if stop_price >= row['Low']:
            if stop_price >= row['Open']:
                exit_price = row['Open']
                
            else:
                exit_price = stop_price
            
            exit_date = row_name.date()
            break

    if not pd.isna(exit_price):
        exit_price = round(exit_price, 2)

    max_pct = ((max_price - entry_price) / entry_price) * 100
    max_pct = round(max_pct, 2)

    return {'exit_date':exit_date, 
            'exit_price': exit_price, 
            'max_pct': max_pct}


In [6]:
# baseline_stop helper function, calculates the max tolerable loss based on given baseline_stop_pct

def baseline_stop(data, entry_price, baseline_stop_pct): 

    baseline_stop_price = entry_price * (1 - (baseline_stop_pct / 100))

    baseline_loop = trade_loop(data, entry_price, baseline_stop_price)
    baseline_max_pct = baseline_loop['max_pct']

    return baseline_max_pct

In [7]:
# False negative helper function, checks whether stop prevented trade from capturing sufficient portion of fat tail

def false_negative_test(max_pct, baseline_max_pct, rally_pct_threshold, tail_pct_threshold):

    if  baseline_max_pct < 0.01:
        tail_pct = np.nan
    else: 
        tail_pct = (max_pct / baseline_max_pct) * 100
        tail_pct = round(tail_pct, 2)

    if tail_pct > 100:
         tail_pct = 100
    
    if (baseline_max_pct >= rally_pct_threshold) and (tail_pct < tail_pct_threshold):
            return {'bool':True, 
                    'tail_pct': tail_pct}

    return {'bool':False, 
            'tail_pct': tail_pct}

In [8]:
# Position sizing helper function

def position_sizing(entry_price, stop_price, max_pct, account_size, portfolio_risk):

    dollar_risk = account_size * portfolio_risk
    trade_risk = 1 - (stop_price / entry_price)

    position_size = dollar_risk / trade_risk

    appreciation_value = position_size * (1 + (max_pct / 100))

    max_trade_ret = appreciation_value - position_size
    
    position_size = round(position_size, 2)
    max_trade_ret = round(max_trade_ret, 2)

    return {'position_size':position_size, 
            'max_trade_ret': max_trade_ret}

In [9]:
# Actual trade result function

def actual_outcome(data, trade_row, baseline_max_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk):

    max_price = trade_row['entry_price']
    
    date = pd.to_datetime(trade_row['entry_date']) + pd.offsets.BusinessDay(1)
    
    if not pd.isna(trade_row['exit_date']):
        for row_name, row in data.loc[date:trade_row['exit_date']].iterrows():

            if row['High'] > max_price:
                max_price = row['High']
    else:
        for row_name, row in data.iloc[1:].iterrows():

            if row['High'] > max_price:
                max_price = row['High']

    max_pct = ((max_price - trade_row['entry_price']) / trade_row['entry_price']) * 100
    max_pct = round(max_pct, 2)

    false_negative = false_negative_test(max_pct, baseline_max_pct, rally_pct_threshold, tail_pct_threshold)

    sizing = position_sizing(trade_row['entry_price'], trade_row['stop_price'], max_pct, account_size, portfolio_risk)

    return {'entry_date': trade_row['entry_date'], 
            'ticker': trade_row['ticker'], 
            'entry_price': trade_row['entry_price'],
            'stop_price': trade_row['stop_price'],
            'exit_date': trade_row['exit_date'],
            'exit_price': trade_row['exit_price'],
            'max_pct': max_pct,
            'baseline_max_pct': baseline_max_pct,
            'tail_pct': false_negative['tail_pct'],
            'false_negative': false_negative['bool'],
            'position_sizing': sizing['position_size'], 
            'max_trade_ret': sizing['max_trade_ret'],
            'stop_type': 'Actual Outcome'}

In [10]:
# Initial stop result function

def initial_stop(data, trade_row, baseline_max_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk):
    
    trade = trade_loop(data, trade_row['entry_price'], trade_row['stop_price'])

    false_negative = false_negative_test(trade['max_pct'], baseline_max_pct, rally_pct_threshold, tail_pct_threshold)

    sizing = position_sizing(trade_row['entry_price'], trade_row['stop_price'], trade['max_pct'], account_size, portfolio_risk)

    return {'entry_date': trade_row['entry_date'], 
            'ticker': trade_row['ticker'], 
            'entry_price': trade_row['entry_price'],
            'stop_price': trade_row['stop_price'],
            'exit_date': trade['exit_date'],
            'exit_price': trade['exit_price'],
            'max_pct': trade['max_pct'],
            'baseline_max_pct': baseline_max_pct,
            'tail_pct': false_negative['tail_pct'],
            'false_negative': false_negative['bool'],
            'position_sizing': sizing['position_size'], 
            'max_trade_ret': sizing['max_trade_ret'],
            'stop_type': 'Initial Stop'}

In [11]:
# Percentage stop function

def pct_stop(data, trade_row, stop_pct, baseline_max_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk):

    stop_price = trade_row['entry_price'] * (1 - (stop_pct / 100))
    stop_price = round(stop_price, 2)

    trade = trade_loop(data, trade_row['entry_price'], stop_price)

    false_negative = false_negative_test(trade['max_pct'], baseline_max_pct, rally_pct_threshold, tail_pct_threshold)

    sizing = position_sizing(trade_row['entry_price'], stop_price, trade['max_pct'], account_size, portfolio_risk)

    return {'entry_date': trade_row['entry_date'], 
            'ticker': trade_row['ticker'], 
            'entry_price': trade_row['entry_price'],
            'stop_price': stop_price,
            'exit_date': trade['exit_date'],
            'exit_price': trade['exit_price'],
            'max_pct': trade['max_pct'],
            'baseline_max_pct': baseline_max_pct,
            'tail_pct': false_negative['tail_pct'],
            'false_negative': false_negative['bool'],
            'position_sizing': sizing['position_size'], 
            'max_trade_ret': sizing['max_trade_ret'],
            'stop_type': f'{stop_pct}_pct Stop'}

In [ ]:
# ATR calculation helper function

def atr_calc(data, atr_window):

    # calculate the true range for every row in the yfinance stock price input. 
    data['TR'] = np.maximum(np.abs(data['High'] - data['Low']), 
                            np.abs(data['High'] - data['Close'].shift(1)), 
                            np.abs(data['Low'] - data['Close'].shift(1))) # np.maximum only takes two arrays, this needs to be fixed !!!

    # At iloc[atr_window] we then calculate the first average true range value, which is pd.na up to that point.
    data['ATR'] = np.nan

    atr_date = data.index[atr_window]
    data.loc[atr_date, 'ATR'] = (data['TR'].iloc[1:atr_window + 1].sum()) / atr_window

    # Then from iloc[atr_window +1:] we use Wilder's smoothing ATR formula. Filling ATR all the way to the end date. 
    for i in range(atr_window + 1, len(data)):

        data.loc[data.index[i], 'ATR'] = ((data['ATR'].iloc[i - 1] * (atr_window - 1)) + data['TR'].iloc[i]) / atr_window

    #return data



In [17]:
ticker = 'TSLA'
start = '2024-01-01'
atr_window = 14

tester = yf.download(ticker, start=start, multi_level_index=False, auto_adjust=True, progress=False)

atr_tester = atr_calc(tester, atr_window)

tester

,Close,High,Low,Open,Volume,TR,ATR
Date,,,,,,,
2024-01-02,248.419998,251.250000,244.410004,250.080002,104654200,NaN,NaN
2024-01-03,238.449997,245.679993,236.320007,244.979996,121082600,9.359985,NaN
2024-01-04,237.929993,242.699997,237.729996,239.250000,102629300,4.970001,NaN
2024-01-05,237.490005,240.119995,234.899994,236.860001,92488900,5.220001,NaN
2024-01-08,240.449997,241.250000,235.300003,236.139999,85166600,5.949997,NaN
...,...,...,...,...,...,...,...
2026-05-28,442.100006,443.959991,436.299988,437.619995,32435000,7.660004,14.338480
2026-05-29,435.790009,441.070007,428.140015,439.850006,45176800,12.929993,14.237874
2026-06-01,415.880005,429.600006,415.429993,427.489990,44937900,14.170013,14.233026


In [14]:
# ATR stop function

def atr_stop():

    ...

In [15]:
# Stop losses simulation function

def stop_sim(trade_row, stop_pct, baseline_stop_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk):
    
    data = yf.download(trade_row['ticker'], start=trade_row['entry_date'], multi_level_index=False, auto_adjust=True, progress=False)

    baseline_max_pct = baseline_stop(data, trade_row['entry_price'], baseline_stop_pct)

    sim_results = []
    sim_results.append(actual_outcome(data, trade_row, baseline_max_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk))
    sim_results.append(initial_stop(data, trade_row, baseline_max_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk))
    
    for i in stop_pct:
        sim_results.append(pct_stop(data, trade_row, i, baseline_max_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk))

    return sim_results


In [16]:
# Reading trades CSV file, then using it as input for a list of dicts, which stores trade outputs.

trades = pd.read_csv('trades.csv')
stop_pct = [8, 10, 12.5, 15, 20, 25, 30]
baseline_stop_pct = 40
rally_pct_threshold = 30
tail_pct_threshold = 40
account_size = 100000
portfolio_risk = 0.01

results = []

for row_name, row in trades.iterrows():
    results.extend(stop_sim(row, stop_pct, baseline_stop_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk))


In [17]:
# List of nested dicts converted into dataframe

results_df = pd.DataFrame(results)
results_df

,entry_date,ticker,entry_price,stop_price,exit_date,exit_price,max_pct,baseline_max_pct,tail_pct,false_negative,position_sizing,max_trade_ret,stop_type
0,2025-08-12,TSLA,340.84,320.15,2025-08-20,320.05,2.39,46.35,5.16,True,16473.66,393.72,Actual Outcome
1,2025-08-12,TSLA,340.84,320.15,2025-08-20,320.15,2.39,46.35,5.16,True,16473.66,393.72,Initial Stop
2,2025-08-12,TSLA,340.84,313.57,NaN,NaN,46.35,46.35,100.00,False,12498.72,5793.16,8_pct Stop
3,2025-08-12,TSLA,340.84,306.76,NaN,NaN,46.35,46.35,100.00,False,10001.17,4635.54,10_pct Stop
4,2025-08-12,TSLA,340.84,298.23,NaN,NaN,46.35,46.35,100.00,False,7999.06,3707.56,12.5_pct Stop
...,...,...,...,...,...,...,...,...,...,...,...,...,...
292,2026-04-08,STX,495.76,433.79,NaN,NaN,93.34,93.34,100.00,False,8000.00,7467.20,12.5_pct Stop
293,2026-04-08,STX,495.76,421.40,NaN,NaN,93.34,93.34,100.00,False,6667.03,6223.00,15_pct Stop
294,2026-04-08,STX,495.76,396.61,NaN,NaN,93.34,93.34,100.00,False,5000.10,4667.09,20_pct Stop
295,2026-04-08,STX,495.76,371.82,NaN,NaN,93.34,93.34,100.00,False,4000.00,3733.60,25_pct Stop


In [18]:
results_df.loc[results_df['stop_type'] == 'Initial Stop']

,entry_date,ticker,entry_price,stop_price,exit_date,exit_price,max_pct,baseline_max_pct,tail_pct,false_negative,position_sizing,max_trade_ret,stop_type
1,2025-08-12,TSLA,340.84,320.15,2025-08-20,320.15,2.39,46.35,5.16,True,16473.66,393.72,Initial Stop
10,2025-08-26,STX,165.36,151.31,NaN,NaN,408.77,408.77,100.00,False,11769.40,48109.76,Initial Stop
19,2025-08-26,CCL,31.89,29.34,2025-09-29,29.34,1.76,5.57,31.60,False,12505.88,220.10,Initial Stop
28,2025-08-28,FLEX,54.77,50.39,NaN,NaN,169.02,169.02,100.00,False,12504.57,21135.22,Initial Stop
37,2025-08-28,BROS,74.22,66.80,2025-09-05,66.80,0.01,0.01,100.00,False,10002.70,1.00,Initial Stop
46,2025-09-05,MU,131.30,118.18,NaN,NaN,523.51,523.51,100.00,False,10007.62,52390.90,Initial Stop
55,2025-09-11,NET,225.50,202.95,2025-11-17,202.95,15.30,15.30,100.00,False,10000.00,1530.00,Initial Stop
64,2025-09-11,RDDT,261.15,235.03,2025-09-24,235.03,8.35,8.35,100.00,False,9998.09,834.84,Initial Stop
73,2025-09-11,TSLA,367.85,331.08,NaN,NaN,35.61,35.61,100.00,False,10004.08,3562.45,Initial Stop
82,2025-09-15,CCJ,86.23,77.61,2025-11-21,77.61,27.50,56.84,48.38,False,10003.48,2750.96,Initial Stop


In [19]:
# Stop loss analysis using pandas groupby function.

aggregated_data = results_df.groupby(['stop_type'], sort=True).agg(
    trades=('ticker', 'count'),
    false_negatives=('false_negative', 'sum'),
    avg_max_pct=('max_pct', 'mean'), 
    avg_tail_capture=('tail_pct', 'mean'), 
    avg_position_size=('position_sizing', 'mean'), 
    avg_max_ret=('max_trade_ret', 'mean'), 
    total_max_ret=('max_trade_ret', 'sum')
).round(2)

aggregated_data

,trades,false_negatives,avg_max_pct,avg_tail_capture,avg_position_size,avg_max_ret,total_max_ret
stop_type,,,,,,,
10_pct Stop,33,8,64.07,65.47,9999.74,6406.64,211419.20
12.5_pct Stop,33,6,72.55,75.49,7999.65,5803.77,191524.37
15_pct Stop,33,5,76.89,83.31,6666.95,5126.40,169171.24
20_pct Stop,33,4,83.30,86.40,5000.04,4165.11,137448.70
25_pct Stop,33,3,89.56,91.50,4000.06,3582.62,118226.49
30_pct Stop,33,0,95.39,98.37,3333.24,3179.77,104932.26
8_pct Stop,33,12,51.59,53.08,12500.24,6450.04,212851.31
Actual Outcome,33,13,30.89,49.46,10152.98,2977.13,98245.43
Initial Stop,33,9,63.17,64.61,10152.98,6512.34,214907.31


In [ ]:
# Using pandas functionality to output dataframe to CSV

results_df.to_csv('results.csv', index=False)